> 在线讲义：[第 4 章 · 多峰（multimodality）：同一个情况，两种都对的做法](https://wikieden.github.io/vla-from-scratch/lessons/04-multimodality.html) · 本书《VLA 从零到 π0》

# 第 4 章 · 多峰（multimodality）：同一个情况，两种都对的做法

配套讲义：仓库里 `vla-deep-dive/notes/lessons/04-multimodality.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python + NumPy，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 写法 | 跑出来 |
|---|---|---|
| 第 1 步 | `l2_answer([0.05, −0.05], [50, 50])` | 0（撞墙） |
| 第 2 步 | `sample_answer(...)` | 只会抽到 ±0.05 |
| 第 3 步 | `switches(8, commit)` | 独立抽签换峰 3 次；画面提醒 0 次 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np

**函数 `l2_answer`**：L2 训练的模型在这个画面下会输出的值：平均。

In [ ]:
def l2_answer(values, counts):
    """L2 训练的模型在这个画面下会输出的值：平均。"""
    v, c = np.asarray(values, float), np.asarray(counts, float)
    return float((v * c).sum() / c.sum())

**函数 `sample_answer`**：生成模型：按示范里的比例抽一个。

In [ ]:
def sample_answer(values, counts, rng):
    """生成模型：按示范里的比例抽一个。"""
    p = np.asarray(counts, float) / sum(counts)
    return float(rng.choice(values, p=p))

**函数 `switches`**：每次重规划都抽一次方向；commit=True 时画面会提醒"已经选了哪边"（只剩那一座峰）。

In [ ]:
def switches(n_replans, commit, rng):
    """每次重规划都抽一次方向；commit=True 时画面会提醒"已经选了哪边"（只剩那一座峰）。"""
    first = rng.choice([-1, 1])
    picks = [first]
    for _ in range(n_replans - 1):
        picks.append(first if commit else rng.choice([-1, 1]))
    return sum(a != b for a, b in zip(picks, picks[1:]))

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 两种都对，平均起来是错的")
print(f"  50 条 Δy = +0.05、50 条 Δy = −0.05：L2 输出 {l2_answer([0.05, -0.05], [50, 50]):+.2f}（撞上石头）")

In [ ]:
print("\n第二步 不求平均，改成抽一个")
rng = np.random.default_rng(0)
draws = [sample_answer([0.05, -0.05], [50, 50], rng) for _ in range(10)]
print(f"  抽 10 次：{draws}（只会是 ±0.05，不会是 0）")

In [ ]:
print("\n第三步 抽定了别变心")
rng = np.random.default_rng(1)
print(f"  每 5 步独立抽签、重抽 8 次：换峰 {switches(8, False, rng)} 次")
print(f"  画面提醒、重抽 8 次：换峰 {switches(8, True, rng)} 次")

## 3. 代码里才看得到的

1. `switches` 里"画面提醒"被简化成"永远选第一次那边"；真实模型里是分布变成只剩一座峰。
2. `sample_answer` 用固定种子，换种子结果会变，但永远不会是 0。

## 4. 自己改一改

- 把示范比例改成 70 / 30，看 `l2_answer` 输出多少。
- 把重规划次数从 8 改成 20，看独立抽签平均换峰几次。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 左绕对，右绕也对，平均一下撞上去；
> 不求平均抽一个，抽到哪条走哪条；
> 抽定一边别变心，一段之内、画面提醒、接着旧计划。